# 01 · เตรียมข้อมูล (Data Preparation)

**เป้าหมาย:** สร้างตารางข้อมูลที่พร้อมวิเคราะห์ **1 ตาราง** โดย 1 แถว = นักศึกษา 1 คน มีรหัส `id` ข้อมูลส่วนตัว และคำตอบ 15 ข้อแบบตัวเลข

| | ไฟล์ | หน้าที่ |
|---|---|---|
| **Input** | `data/Raw_dataset.xlsx` | ข้อมูลหลัก: ข้อมูลส่วนตัว + คำตอบแบบข้อความ |
| | `data/Encoded_dataset.xlsx` | ใช้ตรวจว่าเราแปลงข้อความเป็นตัวเลขถูกต้อง |
| **Output** | `data/processed/stress_clean.csv` | ตารางที่เตรียมแล้ว ส่งต่อให้ `analysis/stress_fa.R` |

**ขั้นตอน**
1. อ่านข้อมูลและดูหน้าตา
2. ตรวจค่าว่างและชนิดข้อมูล
3. ตั้งชื่อคอลัมน์ใหม่และตัดช่องว่าง
4. แปลงคำตอบข้อความเป็นตัวเลข 1–5
5. ตรวจผลการแปลงกับไฟล์ Encoded
6. เพิ่มคอลัมน์ `id`
7. ตรวจแถวซ้ำ และการตอบค่าเดียวกันทุกข้อ
8. ตรวจขนาดกลุ่ม
9. บันทึกไฟล์และตรวจซ้ำ

> รัน notebook นี้จากโฟลเดอร์ `notebooks/` (path ข้อมูลอ้างอิงเป็น `../data/`)

In [1]:
import pandas as pd

DATA_DIR = "../data"
OUT_PATH = f"{DATA_DIR}/processed/stress_clean.csv"

## 1. อ่านข้อมูลและดูหน้าตา

ก่อนแก้อะไร ต้องรู้ก่อนว่าข้อมูลมีกี่แถว กี่คอลัมน์ และแต่ละคอลัมน์หน้าตาเป็นอย่างไร

In [2]:
raw = pd.read_excel(f"{DATA_DIR}/Raw_dataset.xlsx")
print("ขนาดข้อมูล (แถว, คอลัมน์):", raw.shape)
raw.head()

ขนาดข้อมูล (แถว, คอลัมน์): (610, 19)


,Gender,Age,Marital Status,Stage of study,1) I feel nervous before or during exams,2) I find it difficult to fall asleep because of thinking about studying or exams.,3) I have trouble concentrating during lectures.,4) My mind gets distracted quickly when I start studying.,5) I feel overly nervous or irritable due to the pressure of studying.,6) My mood changes significantly during the exam period.,7) I find the curriculum too difficult for me to grasp,8) I feel helpless when materials accumulate or university projects,9) Tuition fees or living costs are a psychological burden on me.,10) Financial pressures affect my concentration and academic level.,11) I feel isolated or lonely within the university.,12) I find it difficult to make friends or have positive relationships with colleagues.,13) I feel that my relationship with some professors is causing me psychological pressure.,14) I feel sad or lose motivation to continue studying.,15) Suffer from insomnia or difficulty sleeping due to constant thinking about university.
0,Female,24,Single,Stage 4,Always,Always,Sometimes,Sometimes,Always,Always,Always,Sometimes,Always,Always,Always,Sometimes,Sometimes,Sometimes,Always
1,Male,22,Single,Stage 4,Always,Often,Sometimes,Often,Rarely,Rarely,Never,Rarely,Never,Never,Never,Never,Never,Never,Never
2,Male,21,Single,Stage 4,Sometimes,Sometimes,Sometimes,Never,Never,Rarely,Rarely,Never,Rarely,Rarely,Sometimes,Rarely,Never,Never,Never
3,Female,20,Single,Stage 4,Sometimes,Never,Rarely,Rarely,Never,Never,Sometimes,Often,Rarely,Rarely,Sometimes,Never,Sometimes,Never,Never
4,Male,21,Single,Stage 2,Rarely,Never,Never,Often,Always,Often,Sometimes,Always,Never,Never,Rarely,Never,Rarely,Sometimes,Never


In [3]:
# ดูชื่อคอลัมน์ทั้งหมด: ใช้ repr() เพื่อให้เห็นช่องว่างที่ซ่อนอยู่
for i, col in enumerate(raw.columns):
    print(i, repr(col))

0 'Gender '
1 'Age'
2 'Marital Status'
3 'Stage of study'
4 '1) I feel nervous before or during exams'
5 '2) I find it difficult to fall asleep because of thinking about studying or exams.'
6 '3) I have trouble concentrating during lectures.'
7 '4) My mind gets distracted quickly when I start studying.'
8 '5) I feel overly nervous or irritable due to the pressure of studying.'
9 '6) My mood changes significantly during the exam period.'
10 '7) I find the curriculum too difficult for me to grasp'
11 '8) I feel helpless when materials accumulate or university projects'
12 '9) Tuition fees or living costs are a psychological burden on me.'
13 '10) Financial pressures affect my concentration and academic level.'
14 '11) I feel isolated or lonely within the university.'
15 '12) I find it difficult to make friends or have positive relationships with colleagues.'
16 '13) I feel that my relationship with some professors is causing me psychological pressure.'
17 '14) I feel sad or lose motivati

**สิ่งที่สังเกตได้**
- 4 คอลัมน์แรกเป็นข้อมูลส่วนตัว ส่วนคอลัมน์ที่ 5–19 เป็นคำตอบ 15 ข้อ
- ชื่อคอลัมน์ `'Gender '` มี**ช่องว่างต่อท้าย** ถ้าพิมพ์ `raw["Gender"]` จะหาไม่เจอ
- ชื่อคอลัมน์คำถามยาวเป็นประโยค ใช้ในโค้ดลำบาก

ข้อ 2 และ 3 จะแก้ในขั้นที่ 3

## 2. ตรวจค่าว่างและชนิดข้อมูล

In [4]:
print("จำนวนค่าว่างทั้งหมด:", raw.isna().sum().sum())
raw.dtypes.value_counts()

จำนวนค่าว่างทั้งหมด: 0


str      18
int64     1
Name: count, dtype: int64

ไม่มีค่าว่างเลย ไม่ต้องตัดแถวหรือเติมค่า คอลัมน์ส่วนใหญ่เป็นข้อความ มีแค่อายุที่เป็นตัวเลข

## 3. ตั้งชื่อคอลัมน์ใหม่และตัดช่องว่าง

ตั้งชื่อสั้น ๆ ให้ใช้ในโค้ดง่าย และตัดช่องว่างหน้าหลังข้อความทุกช่อง (`str.strip()`) กันปัญหาแบบ `'Female'` กับ `'Female '` ถูกนับเป็นคนละค่า

In [5]:
ITEMS = [f"Q{i}" for i in range(1, 16)]          # Q1, Q2, ..., Q15

df = raw.copy()                                   # ทำงานกับสำเนา ไม่แก้ข้อมูลดิบ
df.columns = ["gender", "age", "marital", "stage"] + ITEMS

text_cols = ["gender", "marital", "stage"] + ITEMS
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

df.head(3)

,gender,age,marital,stage,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15
0,Female,24,Single,Stage 4,Always,Always,Sometimes,Sometimes,Always,Always,Always,Sometimes,Always,Always,Always,Sometimes,Sometimes,Sometimes,Always
1,Male,22,Single,Stage 4,Always,Often,Sometimes,Often,Rarely,Rarely,Never,Rarely,Never,Never,Never,Never,Never,Never,Never
2,Male,21,Single,Stage 4,Sometimes,Sometimes,Sometimes,Never,Never,Rarely,Rarely,Never,Rarely,Rarely,Sometimes,Rarely,Never,Never,Never


## 4. แปลงคำตอบข้อความเป็นตัวเลข 1–5

ก่อนแปลง ต้องดูก่อนว่าคำตอบมีกี่แบบ ถ้ามีคำสะกดผิดหรือคำตอบแปลก ๆ จะได้เห็นตั้งแต่ตอนนี้

In [6]:
answers = pd.Series(df[ITEMS].values.ravel())    # รวมคำตอบทุกข้อเป็นคอลัมน์เดียว
answers.value_counts()

Sometimes    2237
Always       2125
Never        1697
Often        1671
Rarely       1420
Name: count, dtype: int64

มีแค่ 5 แบบตรงตามแบบสอบถาม จึงสร้างตารางแปลง (dictionary) แล้วใช้ `.map()` แปลงทีละคอลัมน์

In [7]:
SCALE = {"Never": 1, "Rarely": 2, "Sometimes": 3, "Often": 4, "Always": 5}

for col in ITEMS:
    df[col] = df[col].map(SCALE)

# ถ้ามีคำตอบที่ไม่อยู่ใน SCALE จะกลายเป็นค่าว่าง (NaN) จึงตรวจซ้ำ
print("ค่าว่างหลังแปลง:", df[ITEMS].isna().sum().sum())
print("ค่าต่ำสุด–สูงสุด:", df[ITEMS].min().min(), "–", df[ITEMS].max().max())
assert df[ITEMS].notna().all().all(), "มีคำตอบที่แปลงไม่ได้"

df[ITEMS] = df[ITEMS].astype(int)
df.head(3)

ค่าว่างหลังแปลง: 0
ค่าต่ำสุด–สูงสุด: 1 – 5


,gender,age,marital,stage,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15
0,Female,24,Single,Stage 4,5,5,3,3,5,5,5,3,5,5,5,3,3,3,5
1,Male,22,Single,Stage 4,5,4,3,4,2,2,1,2,1,1,1,1,1,1,1
2,Male,21,Single,Stage 4,3,3,3,1,1,2,2,1,2,2,3,2,1,1,1


## 5. ตรวจผลการแปลงกับไฟล์ Encoded

ผู้จัดทำข้อมูลมีไฟล์ Encoded ที่แปลงเป็นตัวเลขไว้แล้ว เราจึงเทียบผลของเรากับไฟล์นั้น**ทีละช่อง** ถ้าตรงกันทุกช่อง แปลว่า
1. วิธีแปลงของเราถูกต้อง
2. ลำดับแถวของสองไฟล์ตรงกัน

In [8]:
encoded = pd.read_excel(f"{DATA_DIR}/Encoded_dataset.xlsx")
print("ขนาดไฟล์ Encoded:", encoded.shape)

same = df[ITEMS].values == encoded.values         # True/False ทีละช่อง
print("จำนวนช่องที่เทียบ:", same.size)
print("ช่องที่ตรงกัน:", same.sum())
print("ช่องที่ไม่ตรงกัน:", (~same).sum())

assert same.all(), "ผลการแปลงไม่ตรงกับไฟล์ Encoded"   # หยุดทันทีถ้าไม่ตรง

ขนาดไฟล์ Encoded: (610, 15)
จำนวนช่องที่เทียบ: 9150
ช่องที่ตรงกัน: 9150
ช่องที่ไม่ตรงกัน: 0


In [9]:
# ดูตัวอย่าง 3 คนแรก: เทียบข้อความเดิม → ตัวเลขของเรา → ตัวเลขในไฟล์ Encoded
compare = pd.DataFrame({
    "ข้อความ (Raw)":  raw.iloc[:3, 4:8].astype(str).agg(", ".join, axis=1),
    "ที่เราแปลง":       df[ITEMS[:4]].head(3).astype(str).agg(", ".join, axis=1),
    "ไฟล์ Encoded":    encoded.iloc[:3, :4].astype(str).agg(", ".join, axis=1),
})
compare

,ข้อความ (Raw),ที่เราแปลง,ไฟล์ Encoded
0,"Always, Always, Sometimes, Sometimes","5, 5, 3, 3","5, 5, 3, 3"
1,"Always, Often, Sometimes, Often","5, 4, 3, 4","5, 4, 3, 4"
2,"Sometimes, Sometimes, Sometimes, Never","3, 3, 3, 1","3, 3, 3, 1"


## 6. เพิ่มคอลัมน์ `id`

ข้อมูลต้นฉบับไม่มีรหัสผู้ตอบ จึงกำหนด `id` = 1 ถึง 610 ตามลำดับแถว ทุกขั้นหลังจากนี้อ้างถึงนักศึกษาแต่ละคนด้วย `id` ได้

In [10]:
df.insert(0, "id", range(1, len(df) + 1))
df.head(3)

,id,gender,age,marital,stage,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15
0,1,Female,24,Single,Stage 4,5,5,3,3,5,5,5,3,5,5,5,3,3,3,5
1,2,Male,22,Single,Stage 4,5,4,3,4,2,2,1,2,1,1,1,1,1,1,1
2,3,Male,21,Single,Stage 4,3,3,3,1,1,2,2,1,2,2,3,2,1,1,1


## 7. ตรวจแถวซ้ำ

In [11]:
dup_answers = df.duplicated(subset=ITEMS, keep=False)               # คำตอบ 15 ข้อซ้ำกัน
dup_all     = df.drop(columns="id").duplicated(keep=False)          # ซ้ำทั้งแถว (ไม่นับ id)

print("แถวที่คำตอบ 15 ข้อซ้ำกับแถวอื่น:", dup_answers.sum())
print("แถวที่ซ้ำทั้งแถว:", dup_all.sum())
df[dup_answers]

แถวที่คำตอบ 15 ข้อซ้ำกับแถวอื่น: 3
แถวที่ซ้ำทั้งแถว: 0


,id,gender,age,marital,stage,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15
49,50,Female,24,Single,Stage 4,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
304,305,Female,20,Single,Stage 2,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
597,598,Male,23,Single,Stage 2,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5


มีนักศึกษา **3 คน** (id 50, 305, 598) ที่ตอบ 15 ข้อเหมือนกันทุกข้อ แต่**เพศ อายุ หรือชั้นปีต่างกัน** จึงเป็นคนละคน ไม่ใช่ข้อมูลที่บันทึกซ้ำ

แต่ทั้ง 3 คนตอบ **5 (เป็นประจำ) ทุกข้อ** ซึ่งชวนให้สงสัยว่าตั้งใจตอบหรือเปล่า จึงตรวจต่อว่ามีใครตอบค่าเดียวกันทุกข้อบ้าง

### 7.1 ตรวจการตอบค่าเดียวกันทุกข้อ (straight-lining)

ในแบบสอบถาม ผู้ตอบบางคนอาจเลือกคำตอบเดิมทุกข้อโดยไม่อ่านคำถาม ทำให้ข้อมูลคลาดเคลื่อน วิธีตรวจคือนับว่าแต่ละคนใช้คำตอบกี่แบบ ถ้าใช้แบบเดียวทั้ง 15 ข้อ ถือว่าตอบค่าเดียวกันทุกข้อ

In [12]:
n_unique = df[ITEMS].nunique(axis=1)              # แต่ละคนใช้คำตอบกี่แบบ
straight = n_unique == 1

print("ตอบค่าเดียวกันทุกข้อ:", straight.sum(), "คน",
      f"({straight.mean() * 100:.1f}% ของทั้งหมด)")
df.loc[straight, ["id", "gender", "age", "stage", "Q1"]].rename(columns={"Q1": "ค่าที่ตอบทุกข้อ"})

ตอบค่าเดียวกันทุกข้อ: 5 คน (0.8% ของทั้งหมด)


,id,gender,age,stage,ค่าที่ตอบทุกข้อ
49,50,Female,24,Stage 4,5
72,73,Male,22,Stage 4,3
304,305,Female,20,Stage 2,5
581,582,Male,20,Stage 4,2
597,598,Male,23,Stage 2,5


**การตัดสินใจ: เก็บไว้ทั้ง 5 คน** เพราะ
- คิดเป็นแค่ 0.8% ของข้อมูล แทบไม่มีผลต่อผลวิเคราะห์
- ไม่มีหลักฐานว่าตอบไม่ตั้งใจ คนที่เครียดมากจริงอาจตอบ “เป็นประจำ” ทุกข้อ และคนที่ไม่เครียดเลยอาจตอบ “ไม่เคย” ทุกข้อ
- ตัดออกแล้วจะเหลือ 605 คน ไม่ตรงกับขนาดตัวอย่างที่ผู้จัดทำรายงาน

แต่บันทึกไว้เป็นข้อจำกัดของข้อมูล ถ้าต้องการตรวจละเอียดขึ้น ลองตัด 5 คนนี้ออกแล้ววิเคราะห์ซ้ำเพื่อดูว่าผลเปลี่ยนหรือไม่ (sensitivity analysis)

## 8. ตรวจขนาดกลุ่ม

ดูว่าแต่ละกลุ่มที่จะนำไปเปรียบเทียบมีคนพอหรือไม่

In [13]:
for col in ["gender", "marital", "stage"]:
    print(df[col].value_counts().to_string(), "\n")

gender
Female    368
Male      242 

marital
Single     547
Married     63 

stage
Stage 3           180
Stage 4           176
Stage 2           174
Stage 1            73
Master student      6
PhD student         1 


In [14]:
df["age"].describe().round(1)

count    610.0
mean      22.3
std        3.4
min       17.0
25%       20.0
50%       22.0
75%       23.0
max       51.0
Name: age, dtype: float64

**สิ่งที่ต้องตัดสินใจ:** นักศึกษา ป.โท มี 6 คน และ ป.เอก มี 1 คน น้อยเกินกว่าจะเป็นกลุ่มเปรียบเทียบ ตอนเปรียบเทียบชั้นปีจึงใช้เฉพาะปี 1–4 (603 คน)

ทั้ง 7 คนยัง**เก็บไว้ในตาราง** และยังใช้ในการวิเคราะห์ปัจจัยและการเปรียบเทียบเพศหรือสถานภาพ จะตัดออกเฉพาะตอนเปรียบเทียบชั้นปีในไฟล์ R

## 9. บันทึกไฟล์และตรวจซ้ำ

In [15]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)

# อ่านกลับมาตรวจว่าบันทึกครบ
check = pd.read_csv(OUT_PATH)
print("บันทึกแล้ว:", OUT_PATH)
print("ขนาด:", check.shape)
assert check.shape == (610, 20)
assert (check[ITEMS].values == encoded.values).all()
check.head()

บันทึกแล้ว: ../data/processed/stress_clean.csv
ขนาด: (610, 20)


,id,gender,age,marital,stage,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15
0,1,Female,24,Single,Stage 4,5,5,3,3,5,5,5,3,5,5,5,3,3,3,5
1,2,Male,22,Single,Stage 4,5,4,3,4,2,2,1,2,1,1,1,1,1,1,1
2,3,Male,21,Single,Stage 4,3,3,3,1,1,2,2,1,2,2,3,2,1,1,1
3,4,Female,20,Single,Stage 4,3,1,2,2,1,1,3,4,2,2,3,1,3,1,1
4,5,Male,21,Single,Stage 2,2,1,1,4,5,4,3,5,1,1,2,1,2,3,1


## สรุป

| ขั้น | ผล |
|---|---|
| ข้อมูลดิบ | 610 แถว × 19 คอลัมน์ ไม่มีค่าว่าง |
| แปลงคำตอบ | 5 ระดับ → 1–5 แปลงได้ครบ |
| ตรวจกับไฟล์ Encoded | ตรงกัน 9,150 / 9,150 ช่อง |
| แถวซ้ำ | 3 คนตอบเหมือนกันทุกข้อ แต่ข้อมูลส่วนตัวต่างกัน → เป็นคนละคน |
| ตอบค่าเดียวทุกข้อ | 5 คน (0.8%) → เก็บไว้ และบันทึกเป็นข้อจำกัด |
| Output | `stress_clean.csv`: 610 แถว × 20 คอลัมน์ (`id`, ข้อมูลส่วนตัว 4, Q1–Q15) |

**ขั้นต่อไป:** `analysis/stress_fa.R` อ่าน `stress_clean.csv` แล้ววิเคราะห์ปัจจัย